# 1. Imports

In [ ]:
# --- vep path bootstrap (auto-added; safe to re-run) ---
import os, sys
from pathlib import Path
def _vep_root():
    cands = []
    _p = globals().get('__vsc_ipynb_file__')           # VSCode exposes the notebook path
    if _p: cands.append(Path(_p).resolve().parent)
    cands.append(Path.cwd().resolve())
    for s in cands:
        for c in [s, *s.parents]:                       # search upward for the repo root
            if (c / 'core').is_dir() and (c / 'classifiers').is_dir():
                return c
    return Path.cwd().resolve()
_root = _vep_root()
os.chdir(_root)                                          # data/ and outputs/ resolve from repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))                      # makes core/classifiers/app importable
# --- end vep path bootstrap ---

import os
from pathlib import Path
import numpy as np 
import matplotlib.pyplot as plt
from core import config
from core.helpers import load_dataset_paths, compute_average_signal, load_preprocessed_signal, get_data, load_blind_signal
from classifiers.CNN_classifier import CNN1D
from core.preprocessing import preprocess_save_all
import warnings
warnings.filterwarnings("ignore", message="The structure of `inputs` doesn't match the expected structure")

# Change working directory to parent directory
import sys
# Locate the repo root (folder containing the `core` package) and make it importable
_root = Path.cwd()
if not (_root / "core").is_dir():
    _root = _root.parent
os.chdir(_root)                       # data/ and outputs/ paths resolve from the repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))    # so `core`, `classifiers`, `app`, ... import in any kernel

# Optional: verify
print(Path.cwd())

# 2. Loading Files

In [ ]:
DEVICES = config.DEVICES
LABELS = config.LABELS
# Preprocessing
TMIN = config.TMIN
TMAX = config.TMAX
SNR_THRESHOLD = config.SNR_THRESHOLD
# Training
LR = config.LR
EPOCHS = config.EPOCHS
BATCHSIZE = config.BATCHSIZE
DROPOUT = config.DROPOUT
L2_LAMBDA = config.L2_LAMBDA



preprocess_save_all(
    devices=DEVICES,    
    normalize=True,
    tmin=TMIN,
    tmax=TMAX,
    SNR_filtering=True,
    do_artifact_removal=False,
    do_dwt_downsampling=True,
    include_blind=True, 
    SNR_threshold=SNR_THRESHOLD
)


all_paths_preprocessed = load_dataset_paths()
blind_files_sorted = load_blind_signal()


for device in DEVICES:
    print(f"\n{device}:")
    for label in ["BC_Only", "BC_and_RGC"]:
        print(f"{label}: {len(all_paths_preprocessed[device][label])}")

print(f"\nBLIND: {len(blind_files_sorted)}")

In [ ]:
# --- vep path bootstrap (auto-added; safe to re-run) ---
import os, sys
from pathlib import Path
def _vep_root():
    cands = []
    _p = globals().get('__vsc_ipynb_file__')           # VSCode exposes the notebook path
    if _p: cands.append(Path(_p).resolve().parent)
    cands.append(Path.cwd().resolve())
    for s in cands:
        for c in [s, *s.parents]:                       # search upward for the repo root
            if (c / 'core').is_dir() and (c / 'classifiers').is_dir():
                return c
    return Path.cwd().resolve()
_root = _vep_root()
os.chdir(_root)                                          # data/ and outputs/ resolve from repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))                      # makes core/classifiers/app importable
# --- end vep path bootstrap ---

from core.helpers import load_dataset_paths, compute_average_signal, load_preprocessed_signal, get_data, load_blind_signal
DEVICES = ["PRIMA_LE_DA", "MP20_LE_DA", "PRIMA_RCS_DA", "MP20_RCS_LA"]
all_paths_preprocessed = load_dataset_paths()
for device in DEVICES:
    print(f"\n{device}:")
    for label in ["BC_Only", "BC_and_RGC"]:
        print(f"{label}: {len(all_paths_preprocessed[device][label])}")

# 3. Classification

## 3.1. Train Prima Test Prima

In [ ]:
X, labels, raw_X, file_list = get_data(device="PRIMA_LE_DA", classes=2)

cnn = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
y_true, y_pred, cv = cnn.fit_cv(X, labels, epochs=EPOCHS, batch_size=BATCHSIZE, n_splits=10)
results = cnn.evaluate(y_true, y_pred, title="PRIMA LE DA")

print("Mean (all folds combined):")
print("\nCross-validation (mean ± SD across folds):")
print(f"Balanced Accuracy: {cv['balanced_accuracy']['mean']:.4f} ± {cv['balanced_accuracy']['sd']:.4f}")
print(f"F1 Score: {cv['f1']['mean']:.4f} ± {cv['f1']['sd']:.4f}")

## 3.2. Train Prima Test MP20

In [ ]:
X_train, y_train, _, _ = get_data(device="PRIMA_LE_DA",  classes=2)
X_test, y_test, _, _ = get_data(device="MP20_LE_DA", classes=2)


cnn = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
y_true, y_pred, model, _ = cnn.fit_train_val(X=X_train, y=y_train, X_val=X_test, y_val=y_test, epochs=EPOCHS, batch_size=BATCHSIZE)
results = cnn.evaluate(y_true, y_pred, title="MP20 LE DA")

#print results
print("Results:")
print(f"Balanced Accuracy: {results['balanced_accuracy']:.4f}")
print(f"F1 Score: {results['f1']:.4f}")

## 3.3. Train Prima Test MP20 RCS light anastheasia

In [ ]:
X_train, y_train, _, _ = get_data(device="PRIMA_LE_DA",  classes=2)
X_test, y_test, _, _ = get_data(device="MP20_RCS_LA", classes=2)

cnn = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
y_true, y_pred, model, _ = cnn.fit_train_val(X=X_train, y=y_train, X_val=X_test, y_val=y_test, epochs=EPOCHS, batch_size=BATCHSIZE)
results = cnn.evaluate(y_true, y_pred, title="MP20 RCS LA")

#print results
print("Results:")
print(f"Balanced Accuracy: {results['balanced_accuracy']:.4f}")
print(f"F1 Score: {results['f1']:.4f}")

## 3.4. Train PRIMA LE DA Test PRIMA RCS DA

In [ ]:
X_train, y_train, _, _ = get_data(device="PRIMA_LE_DA",  classes=2)
X_test, y_test, _, _ = get_data(device="PRIMA_RCS_DA", classes=2)

cnn = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
y_true, y_pred, model, _ = cnn.fit_train_val(X=X_train, y=y_train, X_val=X_test, y_val=y_test, epochs=EPOCHS, batch_size=BATCHSIZE)
results = cnn.evaluate(y_true, y_pred, title="PRIMA RCS DA")

#print results
print("Results:")
print(f"Balanced Accuracy: {results['balanced_accuracy']:.4f}")
print(f"F1 Score: {results['f1']:.4f}")

## 3.5. Train PRIMA LE DA Test RB20 RCS LA

In [ ]:
X_train, y_train, _, _ = get_data(device="PRIMA_LE_DA",  classes=2)
X_test, y_test, _, _ = get_data(device="RB20_RCS_LA", classes=2)

cnn = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
y_true, y_pred, model, _ = cnn.fit_train_val(X=X_train, y=y_train, X_val=X_test, y_val=y_test, epochs=EPOCHS, batch_size=BATCHSIZE)
results = cnn.evaluate(y_true, y_pred, title="RB20 RCS LA")

#print results
print("Results:")
print(f"Balanced Accuracy: {results['balanced_accuracy']:.4f}")
print(f"F1 Score: {results['f1']:.4f}")

## 3.6. Train PRIMA LE DA Test ALL other conditions

In [ ]:
X_train, y_train, _, _ = get_data(device="PRIMA_LE_DA",  classes=2)
X_test, y_test, _, _ = get_data(device="TEST_ALL", classes=2)

cnn = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
y_true, y_pred, model, _ = cnn.fit_train_val(X=X_train, y=y_train, X_val=X_test, y_val=y_test, epochs=EPOCHS, batch_size=BATCHSIZE)
results = cnn.evaluate(y_true, y_pred, title="Test All")

#print results
print("Results:")
print(f"Balanced Accuracy: {results['balanced_accuracy']:.4f}")
print(f"F1 Score: {results['f1']:.4f}")

# Plot all CMs

In [ ]:
# --- Combined Confusion Matrix Plot: PRIMA / MP20 / MP20* (2 classes) ---
LABELFONTSIZE = 13
configs = [
    ("PRIMA_RCS_DA",                "PRIMA RCS DA", "train PRIMA → test PRIMA_RCS_DA"),
    ("MP20_LE_DA",                  "MP20 LE DA",   "train PRIMA → test MP20_LE_DA"),
    ("MP20_RCS_LA",                 "MP20 RCS LA",  "train PRIMA → test MP20_RCS_LA"),
    ("RB20_RCS_LA",                 "RB20 RCS LA",  "train PRIMA → test RB20_RCS_LA"),
]

cms = []
results = []
class_labels = None

for device, label, mode in configs:
    if device == "PRIMA_LE_DA":
        X, y, _, _ = get_data(device="PRIMA_LE_DA", classes=2)
        cnn_tmp = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
        y_true_tmp, y_pred_tmp, _ = cnn_tmp.fit_cv(X, y, epochs=EPOCHS, batch_size=BATCHSIZE, n_splits=10)
    else:
        X_train, y_train, _, _ = get_data(device="PRIMA_LE_DA", classes=2)
        X_test, y_test, _, _ = get_data(device=device, classes=2)
        cnn_tmp = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
        y_true_tmp, y_pred_tmp, _, _ = cnn_tmp.fit_train_val(
            X=X_train, y=y_train, X_val=X_test, y_val=y_test,
            epochs=EPOCHS, batch_size=BATCHSIZE
        )
    results_tmp = cnn_tmp.evaluate(y_true_tmp, y_pred_tmp, verbose=False)
    results.append(results_tmp)
    cms.append((label, results_tmp["confusion_matrix"]))
    if class_labels is None:
        class_labels = list(cnn_tmp.label_encoder)

# Shorten class labels for display
display_labels = [lbl.replace("_Only", "").replace("_and_", " + ") for lbl in class_labels]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.subplots_adjust(left=0.07, right=0.88, wspace=0.35)

vmin, vmax = 0.0, 1.0
cmap = "viridis"

for ax, (title, cm) in zip(axes, cms):
    im = ax.imshow(cm, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")

    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            val = cm[row, col]
            text_color = "white" if val < 0.6 else "black"
            ax.text(col, row, f"{val:.2f}", ha="center", va="center",
                    fontsize=12, color=text_color, fontweight="bold")

    ax.set_xticks(range(len(display_labels)))
    ax.set_yticks(range(len(display_labels)))
    ax.set_xticklabels(display_labels, fontsize=LABELFONTSIZE)
    ax.set_yticklabels(display_labels, fontsize=LABELFONTSIZE)
    ax.set_xlabel("Predicted", fontsize=LABELFONTSIZE)
    ax.set_title(title, fontsize=14, fontweight="bold")

# Only the leftmost subplot gets the "True" y-label; others hide y tick labels
axes[0].set_ylabel("True", fontsize=LABELFONTSIZE)
for ax in axes[1:]:
    ax.set_yticklabels([])

# Dedicated colorbar axes to the right of all subplots
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Proportion", fontsize=LABELFONTSIZE)
plt.savefig("outputs/figures/CMs_Test_Detail.png", dpi=300, bbox_inches="tight")
plt.show()

#print("BALANCED ACCURACY PRIMA LE DA:", results[0]['balanced_accuracy'])
print("BALANCED ACCURACY PRIMA RCS DA:", results[0]['balanced_accuracy'])
print("BALANCED ACCURACY MP20 LE DA:", results[1]['balanced_accuracy'])
print("BALANCED ACCURACY MP20 RCS LA:", results[2]['balanced_accuracy'])
print("BALANCED ACCURACY RB20 RCS LA:", results[3]['balanced_accuracy'])

In [ ]:
# --- Combined Confusion Matrix Plot: PRIMA / MP20 / MP20* (2 classes) ---
LABELFONTSIZE = 13
configs = [
    ("PRIMA_LE_DA",                 "PRIMA LE DA",  "within-device CV"),
    ("TEST_ALL",                    "TEST (Mixed)",     "train PRIMA → test TEST_ALL"),
]

cms = []
results = []
class_labels = None

for device, label, mode in configs:
    if device == "PRIMA_LE_DA":
        X, y, _, _ = get_data(device="PRIMA_LE_DA", classes=2)
        cnn_tmp = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
        y_true_tmp, y_pred_tmp, _ = cnn_tmp.fit_cv(X, y, epochs=EPOCHS, batch_size=BATCHSIZE, n_splits=10)
    else:
        X_train, y_train, _, _ = get_data(device="PRIMA_LE_DA", classes=2)
        X_test, y_test, _, _ = get_data(device=device, classes=2)
        cnn_tmp = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
        y_true_tmp, y_pred_tmp, _, _ = cnn_tmp.fit_train_val(
            X=X_train, y=y_train, X_val=X_test, y_val=y_test,
            epochs=EPOCHS, batch_size=BATCHSIZE
        )
    results_tmp = cnn_tmp.evaluate(y_true_tmp, y_pred_tmp, verbose=False)
    cms.append((label, results_tmp["confusion_matrix"]))
    results.append((label, results_tmp))
    if class_labels is None:
        class_labels = list(cnn_tmp.label_encoder)


In [ ]:
# Shorten class labels for display
display_labels = [lbl.replace("_Only", "").replace("_and_", " + ") for lbl in class_labels]
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
fig.subplots_adjust(left=0.07, right=0.88, wspace=0.35)

vmin, vmax = 0.0, 1.0
cmap = "viridis"

for ax, (title, cm) in zip(axes, cms):
    im = ax.imshow(cm, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")

    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            val = cm[row, col]
            text_color = "white" if val < 0.6 else "black"
            ax.text(col, row, f"{val:.2f}", ha="center", va="center",
                    fontsize=12, color=text_color, fontweight="bold")

    ax.set_xticks(range(len(display_labels)))
    ax.set_yticks(range(len(display_labels)))
    ax.set_xticklabels(display_labels, fontsize=LABELFONTSIZE)
    ax.set_yticklabels(display_labels, fontsize=LABELFONTSIZE)
    ax.set_xlabel("Predicted", fontsize=LABELFONTSIZE)
    ax.set_title(title, fontsize=14, fontweight="bold")

# Only the leftmost subplot gets the "True" y-label; others hide y tick labels
axes[0].set_ylabel("True", fontsize=LABELFONTSIZE)
for ax in axes[1:]:
    ax.set_yticklabels([])

# Dedicated colorbar axes to the right of all subplots
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Proportion", fontsize=LABELFONTSIZE)

plt.savefig("outputs/figures/CMs_Train_Test.png", dpi=300, bbox_inches="tight")
plt.show()

print("BALANCED ACCURACY PRIMA LE DA:", results[0][1]['balanced_accuracy'])
print("BALANCED ACCURACY TEST ALL:", results[1][1]['balanced_accuracy'])

# 10-Fold Cross-Validation on Combined Dataset

In [ ]:
# First, preprocess all devices
print("=" * 60)
print("PREPROCESSING ALL DEVICES")
print("=" * 60)

ALL_DEVICES = ["PRIMA_LE_DA", "MP20_LE_DA", "PRIMA_RCS_DA", "MP20_RCS_LA", "RB20_RCS_LA"]

preprocess_save_all(
    devices=ALL_DEVICES,
    normalize=True,
    tmin=TMIN,
    tmax=TMAX,
    SNR_filtering=True,
    do_artifact_removal=False,
    do_dwt_downsampling=True,
    include_blind=False,
    SNR_threshold=SNR_THRESHOLD
)

# Verify preprocessing
all_paths_preprocessed = load_dataset_paths()
print("\n" + "=" * 60)
print("PREPROCESSED DATA SUMMARY")
print("=" * 60)
for device in ALL_DEVICES:
    print(f"\n{device}:")
    for label in ["BC_Only", "BC_and_RGC"]:
        print(f"  {label}: {len(all_paths_preprocessed[device][label])}")

# =============================================================================
# Load Combined Dataset
# =============================================================================
print("\n" + "=" * 60)
print("LOADING COMBINED DATASET")
print("=" * 60)

# Collect all data from all devices
X_all = []
y_all = []
device_labels = []  # Track which device each sample came from
file_list_all = []

for device in ALL_DEVICES:
    X_device, y_device, _, files_device = get_data(device=device, classes=2)
    
    X_all.extend(X_device)
    y_all.extend(y_device)
    device_labels.extend([device] * len(X_device))
    file_list_all.extend(files_device)
    
    print(f"{device}: {len(X_device)} samples")

# Convert to arrays
X_all = np.array(X_all)
y_all = np.array(y_all)
device_labels = np.array(device_labels)

print(f"\n{'='*60}")
print(f"TOTAL COMBINED DATASET")
print(f"{'='*60}")
print(f"Total samples: {len(X_all)}")
unique_labels, counts = np.unique(y_all, return_counts=True)
for label, count in zip(unique_labels, counts):
    print(f"  {label}: {count}")


In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
# =============================================================================
# 10-Fold Cross-Validation on Combined Dataset
# =============================================================================
print("\n" + "=" * 60)
print("10-FOLD CROSS-VALIDATION - COMBINED DATASET")
print("=" * 60)

cnn_combined = CNN1D(
    regularization="l2",
    learning_rate=LR,
    l2_lambda=L2_LAMBDA,
    dropout_rate=DROPOUT
)

y_true, y_pred, cv_results = cnn_combined.fit_cv(
    X_all, y_all,
    epochs=EPOCHS,
    batch_size=BATCHSIZE,
    n_splits=10
)

# Calculate confusion matrix
class_labels = list(cnn_combined.label_encoder)
cm_combined = confusion_matrix(y_true, y_pred, labels=class_labels, normalize="true")

print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)
print("\nMean ± SD across 10 folds:")
print(f"Balanced Accuracy: {cv_results['balanced_accuracy']['mean']:.4f} ± {cv_results['balanced_accuracy']['sd']:.4f}")
print(f"F1 Score: {cv_results['f1']['mean']:.4f} ± {cv_results['f1']['sd']:.4f}")
print(f"Accuracy: {cv_results['accuracy']['mean']:.4f} ± {cv_results['accuracy']['sd']:.4f}")

print("\nPer-fold results:")
print(f"{'Fold':<6} {'Accuracy':>10} {'Bal. Acc':>10} {'F1':>10}")
print("-" * 40)
for i in range(10):
    print(f"{i+1:<6} {cv_results['accuracy']['per_fold'][i]:>10.4f}"
          f"{cv_results['balanced_accuracy']['per_fold'][i]:>10.4f}"
          f"{cv_results['f1']['per_fold'][i]:>10.4f}")


In [ ]:
# =============================================================================
# Confusion Matrix Visualization
# =============================================================================

# Shorten class labels for display
display_labels = [lbl.replace("_Only", "").replace("_and_", " + ") for lbl in class_labels]

fig, ax = plt.subplots(1, 1, figsize=(6, 5))

vmin, vmax = 0.0, 1.0
cmap = "viridis"

im = ax.imshow(cm_combined, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")

for row in range(cm_combined.shape[0]):
    for col in range(cm_combined.shape[1]):
        val = cm_combined[row, col]
        text_color = "white" if val < 0.6 else "black"
        ax.text(col, row, f"{val:.2f}", ha="center", va="center",
                fontsize=12, color=text_color, fontweight="bold")

ax.set_xticks(range(len(display_labels)))
ax.set_yticks(range(len(display_labels)))
#ax.set_xticklabels(display_labels, fontsize=LABELFONTSIZE)
#ax.set_yticklabels(display_labels, fontsize=LABELFONTSIZE)
#ax.set_xlabel("Predicted", fontsize=LABELFONTSIZE)
#ax.set_ylabel("True", fontsize=LABELFONTSIZE)
ax.set_title("10-fold cross validation (all devices)", fontsize=14, fontweight="bold")

# Add colorbar directly to the axes (recommended approach)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
#cbar.set_label("Proportion", fontsize=LABELFONTSIZE)

plt.tight_layout()
plt.show()

# Calculate overall metrics
overall_ba = balanced_accuracy_score(y_true, y_pred)
overall_f1 = f1_score(y_true, y_pred, average='weighted')
overall_acc = accuracy_score(y_true, y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix
#=============================================================================
# Per-Device Performance Analysis
# =============================================================================
print("\n" + "=" * 60)
print("PER-DEVICE PERFORMANCE")
print("=" * 60)

# Convert predictions back to same encoding as y_all for consistent indexing
y_true_encoded = cnn_combined._encode_labels(y_true)
y_pred_encoded = cnn_combined._encode_labels(y_pred)

# Calculate per-device metrics
device_metrics = {}
for device in ALL_DEVICES:
    device_mask = device_labels == device
    
    if device_mask.sum() == 0:
        continue
    
    y_true_device = y_true_encoded[device_mask]
    y_pred_device = y_pred_encoded[device_mask]
    
    device_metrics[device] = {
        'n_samples': int(device_mask.sum()),
        'accuracy': accuracy_score(y_true_device, y_pred_device),
        'balanced_accuracy': balanced_accuracy_score(y_true_device, y_pred_device),
        'f1': f1_score(y_true_device, y_pred_device, average='weighted'),
    }

# Print per-device results
print(f"\n{'Device':<20} {'N':<8} {'Accuracy':>10} {'Bal. Acc':>10} {'F1':>10}")
print("-" * 60)
for device in ALL_DEVICES:
    if device in device_metrics:
        m = device_metrics[device]
        print(f"{device:<20} {m['n_samples']:<8} "
              f"{m['accuracy']:>10.4f} "
              f"{m['balanced_accuracy']:>10.4f} "
              f"{m['f1']:>10.4f}")

# Calculate statistics across devices
ba_values = [m['balanced_accuracy'] for m in device_metrics.values()]
acc_values = [m['accuracy'] for m in device_metrics.values()]
f1_values = [m['f1'] for m in device_metrics.values()]

print("-" * 60)
print(f"{'Mean ± SD':<20} {'':<8} "
      f"{np.mean(acc_values):>10.4f} "
      f"{np.mean(ba_values):>10.4f} "
      f"{np.mean(f1_values):>10.4f}")
print(f"{'':30} "
      f"±{np.std(acc_values, ddof=1):>9.4f} "
      f"±{np.std(ba_values, ddof=1):>9.4f} "
      f"±{np.std(f1_values, ddof=1):>9.4f}")

# =============================================================================
# Per-Device Confusion Matrices
# =============================================================================
print("\n" + "=" * 60)
print("PER-DEVICE CONFUSION MATRICES")
print("=" * 60)

LABELFONTSIZE = 13
n_devices = len(ALL_DEVICES)

fig, axes = plt.subplots(1, n_devices, figsize=(4*n_devices, 4))
if n_devices == 1:
    axes = [axes]

fig.subplots_adjust(left=0.05, right=0.92, wspace=0.35)

# Get class labels (shortened)
class_labels_full = list(cnn_combined.label_encoder)
display_labels = [lbl.replace("_Only", "").replace("_and_", " + ") for lbl in class_labels_full]

vmin, vmax = 0.0, 1.0
cmap = "viridis"

for ax, device in zip(axes, ALL_DEVICES):
    device_mask = device_labels == device
    
    if device_mask.sum() == 0:
        ax.set_visible(False)
        continue
    
    y_true_device = y_true_encoded[device_mask]
    y_pred_device = y_pred_encoded[device_mask]
    
    # Calculate normalized confusion matrix
    cm = confusion_matrix(y_true_device, y_pred_device, normalize='true')
    
    im = ax.imshow(cm, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    
    # Add values to cells
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            val = cm[row, col]
            text_color = "white" if val < 0.6 else "black"
            ax.text(col, row, f"{val:.2f}", ha="center", va="center",
                   fontsize=11, color=text_color, fontweight="bold")
    
    ax.set_xticks(range(len(display_labels)))
    ax.set_yticks(range(len(display_labels)))
    ax.set_xticklabels(display_labels, fontsize=LABELFONTSIZE)
    ax.set_yticklabels(display_labels, fontsize=LABELFONTSIZE)
    ax.set_xlabel("Predicted", fontsize=LABELFONTSIZE)
    
    # Title with device name, sample count, and balanced accuracy
    ba = device_metrics[device]['balanced_accuracy']
    n_samples = device_metrics[device]['n_samples']
    ax.set_title(f"{device}\n(n={n_samples}, BA={ba:.3f})", 
                 fontsize=12, fontweight="bold")

# Y-label only on leftmost plot
axes[0].set_ylabel("True", fontsize=LABELFONTSIZE)
for ax in axes[1:]:
    if ax.get_visible():
        ax.set_yticklabels([])

# Shared colorbar
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Proportion", fontsize=LABELFONTSIZE)
plt.show()

print("\n✓ Per-device analysis complete!")

# Store device metrics for later use
cv_results['per_device'] = device_metrics
cv_results['per_device_summary'] = {
    'balanced_accuracy_mean': np.mean(ba_values),
    'balanced_accuracy_std': np.std(ba_values, ddof=1),
    'accuracy_mean': np.mean(acc_values),
    'accuracy_std': np.std(acc_values, ddof=1),
    'f1_mean': np.mean(f1_values),
    'f1_std': np.std(f1_values, ddof=1),
}